#Wikipedia NLP Clustering

##Problem Definition

The goal of this project is to use Natural Language Processing (NLP) to identify the ten most similar people based on their biographies. After cleaning and preprocessing the text, TF-IDF is used to convert biographies into numerical features, and K-Nearest Neighbors (KNN) is used to measure similarity between individuals. Sentiment analysis is performed on the reference person’s biography, and similarity rankings from DBpedia summaries are compared with rankings from full Wikipedia articles. Finally, an interactive notebook allows users to select a person and view their ten closest matches.

##Data Collection

In [1]:
%%capture install_output
%%bash

#Install necessary NLP and Wikipedia API libraries
pip install -q -U textblob wikipedia-api
#Download TextBlob for NLP tasks
python -m textblob.download_corpora

In [44]:
import pandas as pd
import matplotlib.pyplot as plt
import wikipediaapi
import ipywidgets as widgets #Used to create dropdown and button
import urllib.parse #Used to decode special (foreign) characters

from textblob import TextBlob #Tokenization, Counting words, Parts of Speech Sentiment Analysis
from sklearn.feature_extraction.text import CountVectorizer #Understand Bag of Words
from sklearn.feature_extraction.text import TfidfVectorizer #Final Representation
from sklearn.neighbors import NearestNeighbors #Primary algorithm
from IPython.display import display #Pushes widgets to screen

from google.colab import userdata
import os

In [3]:
url = 'https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv'
url

'https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv'

In [4]:
df = pd.read_csv(url)
df

,URI,name,text
0,<http://dbpedia.org/resource/Digby_Morrell>,Digby Morrell,digby morrell born 10 october 1979 is a former...
1,<http://dbpedia.org/resource/Alfred_J._Lewy>,Alfred J. Lewy,alfred j lewy aka sandy lewy graduated from un...
2,<http://dbpedia.org/resource/Harpdog_Brown>,Harpdog Brown,harpdog brown is a singer and harmonica player...
3,<http://dbpedia.org/resource/Franz_Rottensteiner>,Franz Rottensteiner,franz rottensteiner born in waidmannsfeld lowe...
4,<http://dbpedia.org/resource/G-Enka>,G-Enka,henry krvits born 30 december 1974 in tallinn ...
...,...,...,...
42781,<http://dbpedia.org/resource/Motoaki_Takenouchi>,Motoaki Takenouchi,motoaki takenouchi born july 8 1967 saitama pr...
42782,<http://dbpedia.org/resource/Alan_Judge_(footb...,"Alan Judge (footballer, born 1960)",alan graham judge born 14 may 1960 is a retire...
42783,<http://dbpedia.org/resource/Eduardo_Lara>,Eduardo Lara,eduardo lara lozano born 4 september 1959 in c...
42784,<http://dbpedia.org/resource/Tatiana_Faberg%C3...,Tatiana Faberg%C3%A9,tatiana faberg is an author and faberg scholar...


##Data Cleaning

In [5]:
df.shape

(42786, 3)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42786 entries, 0 to 42785
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   URI     42786 non-null  object
 1   name    42786 non-null  object
 2   text    42786 non-null  object
dtypes: object(3)
memory usage: 1002.9+ KB


###Missing Values

In [7]:
df.isnull().sum()

,0
URI,0
name,0
text,0


###Duplicates

In [8]:
#Duplicate rows
print(f"Duplicate rows: {df.duplicated().sum()}")

#Duplicate URIs
print(f"Duplicate URIs: {df['URI'].duplicated().sum()}")

#Duplicate names
print(f"Duplicate names: {df['name'].duplicated().sum()}")

#Duplicate biographies
print(f"Duplicate biographies: {df['text'].duplicated().sum()}")

Duplicate rows: 0
Duplicate URIs: 0
Duplicate names: 1
Duplicate biographies: 0


In [9]:
#View rows with duplicate names
df[df['name'].duplicated(keep=False)]

,URI,name,text
787,<http://dbpedia.org/resource/James_Grieve_(tra...,author),james grieve born 1934 is an australian transl...
17249,<http://dbpedia.org/resource/Steve_Greenberg_(...,author),steve greenberg december 20 1960 is an america...


In [10]:
#Extract names from the URI
extracted_names = (
    df["URI"]
    #Extract between resource/ and next parenthesis
    .str.extract(r"resource/([^(\n]+)")[0]
    .str.replace("_", " ", regex=False)
    .str.strip()
)

#Identify invalid names
mask = df["name"].str.contains(
    r"^author\)$",
    regex=True,
    na=False
)

#Replace invalid names
df.loc[mask, "name"] = extracted_names[mask]

In [11]:
#Verify Corrections
df[df['name'].str.contains(r'^author\)$', na=False)]

,URI,name,text


###Name Cleaning

In [12]:
#Unique names
print(f"Unique names: {df['name'].nunique():,}")

#Missing values
print(f"Missing names: {df['name'].isnull().sum()}")

#Create working copy
df["clean_name"] = df["name"]

Unique names: 42,786
Missing names: 0


####Remove Extra Whitespace

In [13]:
df['clean_name'] = (
    df['clean_name']
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

####Decode URL-Encoded Characters

In [14]:
df['clean_name'] = df['name'].apply(urllib.parse.unquote)

In [15]:
#Validate Name Cleaning
df[df['name'].str.contains('Javier S', case=False, na=False)][
    ['name', 'clean_name', 'text']
]

,name,clean_name,text
10133,Javier S%C3%A1nchez,Javier Sánchez,javier snchez vicario born 1968 in pamplona sp...
34536,Javier Santiso,Javier Santiso,javier santiso is a young global leader ygl of...


####Check Duplicate Clean Names

In [16]:
print(f"Duplicate clean names: {df['clean_name'].duplicated().sum()}")

Duplicate clean names: 0


In [17]:
df[df['clean_name'].duplicated(keep=False)]

,URI,name,text,clean_name


###Text Cleaning

In [18]:
#Create working copy
df["clean_text"] = df["text"]

#Missing values
print(f"Missing biographies: {df['text'].isnull().sum()}")

#Remove whitespace
print(f"Blank biographies: {(df['text'].str.strip() == '').sum()}")

Missing biographies: 0
Blank biographies: 0


In [19]:
#Calculate length of text
df["text_length"] = df["text"].apply(len)

df["text_length"].describe().transpose()

,text_length
count,42786.000000
mean,1896.548801
std,833.085754
min,1049.000000
25%,1386.000000
50%,1655.000000
75%,2125.000000
max,31850.000000


####Convert to Lowercase

In [20]:
df["clean_text"] = df["clean_text"].str.lower()

####Remove Punctuation

In [21]:
df["clean_text"] = (
    df["clean_text"]
    #Means keep words and spaces, remove everything else
    .str.replace(r"[^\w\s]", "", regex=True)
)

####Remove Extra Whitespaces

In [22]:
df["clean_text"] = (
    df["clean_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

####Lemmatization

In [23]:
#Reduce words to their base form
df["clean_text"] = df["clean_text"].apply(
    lambda text: " ".join(
        word.lemmatize()
        for word in TextBlob(text).words
    )
)

In [24]:
#Compare single biography
person_name = "Digby Morrell"

original = df.loc[
    df["clean_name"] == person_name,
    "text"
].iloc[0]

cleaned = df.loc[
    df["clean_name"] == person_name,
    "clean_text"
].iloc[0]

print("Original:\n")
print(original)

print("\n" + "-"*80 + "\n")

print("Cleaned:\n")
print(cleaned)

Original:

digby morrell born 10 october 1979 is a former australian rules footballer who played with the kangaroos and carlton in the australian football league aflfrom western australia morrell played his early senior football for west perth his 44game senior career for the falcons spanned 19982000 and he was the clubs leading goalkicker in 2000 at the age of 21 morrell was recruited to the australian football league by the kangaroos football club with its third round selection in the 2001 afl rookie draft as a forward he twice kicked five goals during his time with the kangaroos the first was in a losing cause against sydney in 2002 and the other the following season in a drawn game against brisbaneafter the 2003 season morrell was traded along with david teague to the carlton football club in exchange for corey mckernan he played 32 games for the blues before being delisted at the end of 2005 he continued to play victorian football league vfl football with the northern bullants car

##Create Clean DataFrame

In [25]:
df_clean = df[['clean_name', 'clean_text']].copy()
df_clean

,clean_name,clean_text
0,Digby Morrell,digby morrell born 10 october 1979 is a former...
1,Alfred J. Lewy,alfred j lewy aka sandy lewy graduated from un...
2,Harpdog Brown,harpdog brown is a singer and harmonica player...
3,Franz Rottensteiner,franz rottensteiner born in waidmannsfeld lowe...
4,G-Enka,henry krvits born 30 december 1974 in tallinn ...
...,...,...
42781,Motoaki Takenouchi,motoaki takenouchi born july 8 1967 saitama pr...
42782,"Alan Judge (footballer, born 1960)",alan graham judge born 14 may 1960 is a retire...
42783,Eduardo Lara,eduardo lara lozano born 4 september 1959 in c...
42784,Tatiana Fabergé,tatiana faberg is an author and faberg scholar...


#Save as a Parquet

In [45]:
parquet_file = 'wiki.parquet'
parquet_file

'wiki.parquet'

In [46]:
df_clean.to_parquet(parquet_file, index=False)

In [47]:
!ls -l --si {parquet_file}

-rw-r--r-- 1 root root 49M Jul 21 04:24 wiki.parquet


In [48]:
df2 = pd.read_parquet(parquet_file)
df2.shape

(42786, 2)

In [49]:
os.environ["HF_TOKEN"] = userdata.get('hf_token')
_ = os.environ["HF_TOKEN"]
f"{_[:5]} ... {_[-3:]}"

'hf_rP ... BIc'

In [50]:
os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account = os.environ["HF_ACCOUNT"]
hf_account

'stephanie465337'

In [51]:
hf_repo = "Data_Science-21"
os.environ["HF_REPO"] = hf_repo
hf_repo

'Data_Science-21'

In [52]:
!hf auth login --token $HF_TOKEN

Hint: A new version of huggingface_hub (1.24.0) is available! You are using version 1.23.0.
To update, run: hf update
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Colab` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [53]:
%%capture hf_upload
%%bash
hf upload \
  --type dataset \
  ${HF_ACCOUNT}/${HF_REPO} \
  wiki.parquet

In [54]:
print(hf_upload.stdout)

✓ Uploaded
  url: https://huggingface.co/datasets/Stephanie465337/Data_Science-21/commit/dd8a07a5b6991f08b37394313b7ebc2c962d80ee



In [55]:
hf_url = f"https://huggingface.co/datasets/{hf_account}/{hf_repo}/resolve/main/wiki.parquet"
hf_url

'https://huggingface.co/datasets/stephanie465337/Data_Science-21/resolve/main/wiki.parquet'

In [56]:
df3 = pd.read_parquet(hf_url)
df3.shape

(42786, 2)

In [57]:
df3.iloc[:,:5].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42786 entries, 0 to 42785
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   clean_name  42786 non-null  object
 1   clean_text  42786 non-null  object
dtypes: object(2)
memory usage: 668.7+ KB
